# Graph Neural Network for identifying Twitter bots
One of the most compelling reasons to employ GNNs over classical machine learning models, especially for predicting Twitter bots within a network, is their **ability to handle relational data effectively**. Twitter networks, where users (nodes) are connected by various types of interactions (edges), such as retweets and replies, are prime examples of complex graph structures. **Classical machine learning models often struggle with this type of data due to their inability to natively understand the connections and the flow of information within the network.** In contrast, GNNs can directly model these relationships, enabling them to detect subtle patterns indicative of bot-like behavior that classical models might miss.


Versioning:
- Created by Heng Boon Long; Kyaw Khant Zaw; Joshua Lim Wei-En; Lee Ee Fun; ; Lee Gene Ee; Tay Han (in alphabet order)
- Revised by Liu Yan and Wang Qiuhong

# Section 1. Preparation

## Importing Dependencies

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix
import seaborn as sns
from sklearn.metrics import precision_score, recall_score, accuracy_score, f1_score

In [ ]:
# download torch_geometric if haven't done so
!pip install torch_geometric

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.1/63.1 kB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 18.8 MB/s eta 0:00:00


In [ ]:
import torch
import torch.nn.functional as F
from torch import nn
from torch_geometric.nn import GCNConv
from torch_geometric.data import Data

## Loading data

We have already extracted user profile and users' relationship (reply,retweet) from the raw data in twitter-bot 22 https://twibot22.github.io/. In addition, we have generated the twit_embedding based on the content of any tweet related to a user (including the user's own content, retweet, replied tweet). Please refer to the lecture notes for the preprocessing steps that are omitted here fo simplicity.

In [ ]:
from google.colab import drive
### Permit this notebook to access your Google Drive files?
### Please select "Connect to Google Drive", choose your account and select continue
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
### adjust the data directory if needed
### Upload the three data files into your Google Drive
### graph.csv (420.7MB)
### users.csv (820KB))
### bot_clusters_ids.csv (38.1MB)
%cd /content/drive/My Drive/ # please replace it with your own directory where the datasets are located.

In [ ]:
u = pd.read_csv("users.csv")
g = pd.read_csv("graph.csv", float_precision='round_trip') # graph.csv may takes some time to be loaded in google collab

In [ ]:
g.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 315931 entries, 0 to 315930
Data columns (total 4 columns):
 #   Column           Non-Null Count   Dtype  
---  ------           --------------   -----  
 0   source_user_id   315931 non-null  int64  
 1   target_user_id   315931 non-null  float64
 2   relationship     315931 non-null  object 
 3   tweet_embedding  294962 non-null  object 
dtypes: float64(1), int64(1), object(2)
memory usage: 9.6+ MB


In [ ]:
g.head()

,source_user_id,target_user_id,relationship,tweet_embedding
0,20555412,1.445930e+08,Retweet,[ 0.3512645 -0.08518124 0.03606087 -0.206615...
1,2768457999,1.407822e+09,Retweet,[ 1.50224999e-01 3.60594988e-01 1.63409993e-...
2,2768457999,9.390910e+05,Retweet,[-6.35727448e-03 3.19840759e-01 1.26310363e-...
3,2768457999,1.154529e+18,Retweet,[-1.52178392e-01 2.46049643e-01 1.06427386e-...
4,2768457999,3.091246e+08,Retweet,[ 0.46057498 0.21094498 -0.65761 -0.274167...


Note: According to g.head(), we can see that target_user_id does not follow the same format as the source_user_id; and the tweet_embedding is in a string format. So we need to check whether the target_user_id information is available in the users.csv file and we also need to convert target_user_id from float type to int type, same as source_user_id. We further convert embeddings from string to list. The following processing steps serve such a purpose.

In [ ]:
u.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 109425 entries, 0 to 109424
Data columns (total 34 columns):
 #   Column                Non-Null Count   Dtype  
---  ------                --------------   -----  
 0   source_user_id        109425 non-null  int64  
 1   verified              109425 non-null  int64  
 2   url.urls              109425 non-null  float64
 3   description.urls      109425 non-null  float64
 4   description.mentions  109425 non-null  float64
 5   description.hashtags  109425 non-null  float64
 6   description.cashtags  109425 non-null  float64
 7   followers_count       109425 non-null  float64
 8   following_count       109425 non-null  float64
 9   tweet_count           109425 non-null  float64
 10  listed_count          109425 non-null  float64
 11  username_length       109425 non-null  float64
 12  name_length           109425 non-null  float64
 13  description_length    109425 non-null  float64
 14  numDigits_username    109425 non-null  float64
 15  

In [ ]:
u.head()

,source_user_id,verified,url.urls,description.urls,description.mentions,description.hashtags,description.cashtags,followers_count,following_count,tweet_count,...,url_count_tweets,url_ratio,url_max_tweets,time_interval_sd_day,mention_count_tweets,mention_max_tweets,hashtag_count_tweets,hashtags_max_tweets,avg_tweet_length,url_tweet_count
0,22,0,1.0,0.0,0.0,0.0,0.0,16772.0,4562.0,29394.0,...,0.034545,0.00000,0.138439,0.010237,0.021362,0.054509,0.004140,0.072424,137.037371,55.349359
1,59,1,0.0,0.0,4.0,0.0,0.0,9600.0,2790.0,29191.0,...,0.017501,0.00000,0.109994,0.018869,0.008840,0.049304,0.006963,0.063746,138.309138,9.280581
2,76,1,0.0,0.0,1.0,0.0,0.0,19034.0,1539.0,72759.0,...,0.030235,0.00118,0.113362,0.048655,0.009284,0.047087,0.012448,0.045912,138.093520,8.524393
3,150,0,1.0,0.0,1.0,0.0,0.0,14094.0,1931.0,22020.0,...,0.063662,0.00000,0.152344,0.006156,0.008209,0.047662,0.011290,0.053273,140.310942,10.560000
4,187,0,1.0,0.0,0.0,0.0,0.0,2162.0,256.0,7886.0,...,0.063662,0.00000,0.152344,0.006156,0.008209,0.047662,0.011290,0.053273,140.310942,10.560000


In [ ]:
# cannot match target_user_id to user
problematic_rows = []
for index, row in g.iterrows():
    try:
        val = u[u["source_user_id"]==row["target_user_id"]]["source_user_id"].values[0]
    except:
        problematic_rows.append(index)

In [ ]:
len(problematic_rows) # 16766

16766

In [ ]:
# drops rows that cannot match
g = g.drop(problematic_rows, axis=0)

In [ ]:
# Convert the float to int
def convert_float_to_int(x):
    val = u[u["source_user_id"]==x]["source_user_id"].values[0]
    return val

g["target_user_id"] = g["target_user_id"].apply(convert_float_to_int)

In [ ]:
def convert_str_to_list(embedding):
    if (pd.isna(embedding)):
        embedding = str(np.zeros(100))
    t = embedding.replace("  ", ",").replace("\n ", ",").replace("\n", ",").replace(" ", ",")
    return [float(x) for x in t[1:-1].split(",") if x != ""]
g["tweet_embedding"] = g["tweet_embedding"].apply(convert_str_to_list)

# Section 2. Graph Neural Networks
1. Constructing the graph structure (**edge_index**, **edge_weight**, **node_features**, ) where the **nodes are represented by the aggregated tweet embedding + top 10 selected user features**, and **the edge are represented by a reply/retweet relationship between nodes (users)**.
2. Create the pytorch `Data` object.
3. Run `custom_stratified_split` to get training, validation and test masks.
4. Build custom functions for `EarlyStopping`, `train_validate`, `test`.
5. Train and validate on the following GNN models with `Adam Optimizer`, `learning rate = 0.05` and `decay = 0.01`:
   - `Graph Convolutional Network (GCN)`
   - `GraphSAGE`
   - `Graph Attention Network (GAT)`
6. Evaluate model using `accuracy`, `precision`, `recall`, `f1-score` on the test set and plot `confusion matrix`.

###Step 1. Construct graph structure and aggregate (mean) tweet embedding for all users

In [ ]:
# Create a mapping of user IDs to node indices >> change it up to index
user_to_idx = {user_id: idx for idx, user_id in enumerate(u["source_user_id"])}

# Create edge index and edge weight tensors
edge_list = []
edge_weight_dict = {}

for _, row in g.iterrows():
    source_idx = user_to_idx[row['source_user_id']]
    target_idx = user_to_idx[row['target_user_id']]
    idx_pair = (source_idx, target_idx)
    if (idx_pair not in edge_list):
        edge_list.append(idx_pair)
        edge_weight_dict[idx_pair] = 1
    else:
        edge_weight_dict[idx_pair] += 1

edge_weight_list = list(edge_weight_dict.values())

#force it to be contiguous
edge_index = torch.tensor(edge_list, dtype=torch.long).t().contiguous()
edge_weight = torch.tensor(edge_weight_list, dtype=torch.float)

In [ ]:
# Aggregate embeddings of post/reply/retweet for each node
def aggregate_embeddings(user_id):
    user = g[g['source_user_id'] == user_id]
    if user.size != 0:
        embeddings = np.stack(user['tweet_embedding'].values)
    else:
        # if user did not make any tweets
        embeddings = np.zeros((1,100))
    return torch.mean(torch.from_numpy(embeddings), dim=0)

tweet_embedding_tensor = torch.stack([aggregate_embeddings(user_id) for user_id in u["source_user_id"]])
user_features = u[['mention_max_tweets','hashtags_max_tweets','retweet_ratio','url_tweet_count','hashtag_count_tweets','tweet_count',
 'time_interval_sd_day','followers_count','verified','listed_count']]
user_features_tensor = torch.from_numpy(user_features.to_numpy())

In [ ]:
print(f"size of user tensor: {user_features_tensor.shape}")
print(f"size of embedding tensor: {tweet_embedding_tensor.shape}")

node_features_tensor = torch.cat((user_features_tensor, tweet_embedding_tensor), dim=1)
print(f"size of combined node tensor: {node_features_tensor.shape}")

label_tensor = torch.from_numpy(u["label"].values)
print(f"size of label tensor: {label_tensor.shape}")


'''
size of user tensor: torch.Size([109425, 11])
size of embedding tensor: torch.Size([109425, 100])
size of combined node tensor: torch.Size([109425, 111])
size of label tensor: torch.Size([109425])
'''

size of user tensor: torch.Size([109425, 10])
size of embedding tensor: torch.Size([109425, 100])
size of combined node tensor: torch.Size([109425, 110])
size of label tensor: torch.Size([109425])


'\nsize of user tensor: torch.Size([109425, 11])\nsize of embedding tensor: torch.Size([109425, 100])\nsize of combined node tensor: torch.Size([109425, 111])\nsize of label tensor: torch.Size([109425])\n'

In [ ]:
edge_index.shape #torch.Size([2, 132371])

torch.Size([2, 132371])

In [ ]:
edge_index

tensor([[ 9334, 64146, 64146,  ..., 63209, 38609, 38609],
        [29001, 55105,   399,  ..., 30513, 29358, 47585]])

In [ ]:
edge_weight.shape

torch.Size([132371])

In [ ]:
print("mean weight:", edge_weight.mean(),"\nminimum weight:", edge_weight.min(),"\nmaximum weight:", edge_weight.max())

mean weight: tensor(2.2600) 
minimum weight: tensor(1.) 
maximum weight: tensor(691.)


###Step 2. Run custom_stratified_split with 80% trainset, 10% valset and 10% testset
We will be training the data using the train_mask and evaluate it using the val_mask before testing it on the test_mask to get the final performance scores.

In [ ]:
def custom_stratified_split(data, train_ratio, val_ratio, test_ratio):

    # Ensure the ratios sum to 1
    assert train_ratio + val_ratio + test_ratio == 1, "Ratios must sum to 1"

    num_nodes = data.num_nodes
    num_classes = int(torch.max(data.y)) + 1

    # Masks initialization
    train_mask = torch.zeros(num_nodes, dtype=torch.bool)
    val_mask = torch.zeros(num_nodes, dtype=torch.bool)
    test_mask = torch.zeros(num_nodes, dtype=torch.bool)

    for class_idx in range(num_classes):
        # Get indices of nodes in the current class
        class_indices = (data.y == class_idx).nonzero(as_tuple=False).view(-1)
        # Shuffle indices
        class_indices = class_indices[torch.randperm(len(class_indices))]

        # Compute split sizes
        num_train_per_class = int(len(class_indices) * train_ratio)
        num_val_per_class = int(len(class_indices) * val_ratio)

        # Assign masks
        train_mask[class_indices[:num_train_per_class]] = True
        val_mask[class_indices[num_train_per_class:num_train_per_class + num_val_per_class]] = True
        test_mask[class_indices[num_train_per_class + num_val_per_class:]] = True

    return train_mask, val_mask, test_mask

###Step 3. Create pytorch geometric Data object

In [ ]:
# create a PyTorch Geometric data object

# tweet embedding + top 10 user features
data = Data(x=node_features_tensor.float(), y=label_tensor.long(), edge_index=edge_index, edge_weight=edge_weight)

train_mask, val_mask, test_mask = custom_stratified_split(data, train_ratio=0.8, val_ratio=0.1, test_ratio=0.1)

###Step 4. Build custom functions for `EarlyStopping`, `train_validate` and `test`

In [ ]:
class EarlyStopping:
    def __init__(self, patience=3, verbose=False, delta=0, path='checkpoint.pt', trace_func=print):
        self.patience = patience
        self.verbose = verbose
        self.counter = 0
        self.best_score = None
        self.early_stop = False
        self.val_loss_min = np.Inf
        self.delta = delta
        self.path = path
        self.trace_func = trace_func

    def __call__(self, val_loss, model):
        score = -val_loss

        if self.best_score is None:
            self.best_score = score
            self.save_checkpoint(val_loss, model)
        elif score <= self.best_score + self.delta:
            self.counter += 1
            self.trace_func(f'EarlyStopping counter: {self.counter} out of {self.patience}')
            if self.counter >= self.patience:
                self.early_stop = True
        else:
            self.best_score = score
            self.save_checkpoint(val_loss, model)
            self.counter = 0

    def save_checkpoint(self, val_loss, model):
        if self.verbose:
            self.trace_func(f'Validation loss decreased ({self.val_loss_min:.6f} --> {val_loss:.6f}).  Saving model ...')
        self.val_loss_min = val_loss

In [ ]:
def train(model, data, criterion, optimizer, has_edge_weight=True):
    model.train()
    optimizer.zero_grad()
    if has_edge_weight:
        out = model(data.x, data.edge_index, data.edge_weight)
    else:
        out = model(data.x, data.edge_index)
    loss = criterion(out[train_mask], data.y[train_mask])
    loss.backward()
    optimizer.step()
    return loss

def validate(model, data, criterion, optimizer, has_edge_weight=True):
    model.eval()
    with torch.no_grad():
        if has_edge_weight:
            out = model(data.x, data.edge_index, data.edge_weight)
        else:
            out = model(data.x, data.edge_index)
        loss = criterion(out[val_mask], data.y[val_mask])

        # for BCE Loss
        _, pred = torch.max(out, 1)

        # Convert to CPU and numpy for sklearn compatibility
        true_labels = data.y[val_mask].cpu().numpy()
        pred_labels = pred[val_mask].cpu().numpy()

        # Calculate precision, recall, and accuracy
        precision = precision_score(true_labels, pred_labels, average='binary')
        recall = recall_score(true_labels, pred_labels, average='binary')
        accuracy = accuracy_score(true_labels, pred_labels)
        f1 = f1_score(true_labels, pred_labels)

    return loss, precision, recall, accuracy, f1

# ----------------------------------------------- Functions for training and testing --------------------------------------------------------

def train_validate(model, data, criterion, optimizer, epoch, early_stopping_patience, has_edge_weight=True):
    early_stopping = EarlyStopping(patience=early_stopping_patience, verbose=False)
    for epoch in range(epoch):
        loss = train(model, data, criterion, optimizer, has_edge_weight)
        val_loss, val_precision, val_recall, val_acc, val_f1 = validate(model, data, criterion, optimizer, has_edge_weight)
        print(f'Epoch {epoch+1} | Train Loss: {loss:.4f}, Val Loss: {val_loss:.4f}, Val Precision: {val_precision:.4f}, Val Recall: {val_recall:.4f}, Val Acc: {val_acc:.4f}, Val f1: {val_f1:.4f}')

        # Early stopping
        early_stopping(val_loss, model)
        if early_stopping.early_stop:
            print("Early stopping")
            break

def test(model, data, criterion, optimizer, has_edge_weight=True, show_cm=True, show_df=False):
    model.eval()
    with torch.no_grad():
        if has_edge_weight:
            out = model(data.x, data.edge_index, data.edge_weight)
        else:
            out = model(data.x, data.edge_index)
        test_loss = criterion(out[test_mask], data.y[test_mask].long())
        pred = out.argmax(dim=1)

        # Convert to CPU and numpy for sklearn compatibility
        true_labels = data.y[test_mask].cpu().numpy()
        pred_labels = pred[test_mask].cpu().numpy()

        # Calculate precision, recall, and accuracy
        precision = precision_score(true_labels, pred_labels, average='binary')
        recall = recall_score(true_labels, pred_labels, average='binary')
        accuracy = accuracy_score(true_labels, pred_labels)
        f1 = f1_score(true_labels, pred_labels)


    test_pred, test_labels, test_loss, test_precision, test_recall, test_acc, test_f1 = pred[test_mask], data.y[test_mask], test_loss, precision, recall, accuracy, f1
    print(f'Test Loss: {test_loss:.4f}, Test Precision: {test_precision:.4f}, Test Recall: {test_recall:.4f}, Test Acc: {test_acc:.4f}, Test F1: {test_f1:.4f}')

    # set whether to plot confusion matrix
    if (show_cm):
        cm = confusion_matrix(test_labels.cpu().numpy(), test_pred.cpu().numpy())
        # Plot the confusion matrix
        plt.figure(figsize=(10, 7))
        sns.heatmap(cm, annot=True, fmt="d", cmap="Blues")
        plt.xlabel('Predicted labels')
        plt.ylabel('True labels')
        plt.title('Confusion Matrix')
        plt.show()

    # Create a DataFrame with source_user_id, true labels, and predicted labels
    if show_df:
        predicted_id_df = pd.DataFrame({
            'source_user_id': u["source_user_id"][test_mask.cpu().numpy()],
            'true_label': true_labels,
            'predicted_label': pred_labels
        })
        return predicted_id_df

### Step5 GNN Models

####5.1 GCN Model
GCN uses a weighted sum of the features from neighboring nodes to aggregate information.

Several important hyperparameters:
1. in_channels (int) – Size of each input sample, or -1 to derive the size from the first input(s) to the forward method.

2. hidden_channels (int) – Size of each hidden sample.

3. num_layers (int) – Number of message passing layers.

4. out_channels (int, optional) – If not set to None, will apply a final linear transformation to convert hidden node embeddings to output size out_channels. (default: None)

- `Dimsension: num_features`
- `1 GCNConv layer`
- `Batch Normalization`
- `Dropout: 0.3`
- `Feed-forward network with 2 linear layers`

In [ ]:
class GCN1(nn.Module):
    def __init__(self, num_features, num_classes):
        super(GCN1, self).__init__()
        self.bn = torch.nn.BatchNorm1d(num_features)
        self.conv = GCNConv(num_features, num_features)
        self.dropout = nn.Dropout(0.3)
        self.linear1 = torch.nn.Linear(num_features, num_features)
        self.linear2 = torch.nn.Linear(num_features, num_classes)

    def forward(self, x, edge_index, edge_weight):

        x = F.leaky_relu(self.conv(x, edge_index, edge_weight=edge_weight))
        x = self.dropout(x)

        x = self.bn(x)
        x = F.leaky_relu(self.linear1(x))
        x = self.dropout(x)

        x = self.linear2(x)

        return x

In [ ]:
model = GCN1(num_features=node_features_tensor.size(1), num_classes=2)

decay = 0.05
lr = 0.001

criterion = torch.nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=decay)

train_validate(model=model, data=data, criterion=criterion, optimizer=optimizer, epoch=149, early_stopping_patience=3)

In [ ]:
test(model, data, criterion, optimizer, show_cm=True)

#### 5.2 GraphSAGE Model
GraphSAGE introduces a sampling mechanism and multiple types of aggregators (e.g., mean, LSTM, pooling) for combining the features from neighboring nodes. This Instead of aggregating from all neighbors like GCN, GraphSAGE samples a fixed number of neighbors and then aggregates their features. This makes it scalable to large graphs.

GraphSAGE's ability to perform inductive learning and its neighborhood sampling technique make it particularly well-suited for identifying bots within a network like Twitter.

It can also generalize better to unseen nodes not seen in training which aligns with our objective.


Several important hyperparameters:
1. in_channels (int or tuple) – Size of each input sample, or -1 to derive the size from the first input(s) to the forward method. A tuple corresponds to the sizes of source and target dimensionalities.

2. hidden_channels (int) – Size of each hidden sample.

3. num_layers (int) – Number of message passing layers.

4.  out_channels (int, optional) – If not set to None, will apply a final linear transformation to convert hidden node embeddings to output size out_channels. (default: None)

- `Dimsension: 64 and 32`
- `Batch Normalization and feed-forward after input layer`
- `1 GraphSAGE layer` with `mean pooling`
- `Batch Normalization`
- `Dropout: 0.2`
- `Feed-forward network with 2 linear layers`

In [ ]:
from torch_geometric.nn import SAGEConv

class GraphSAGE2(nn.Module):
    def __init__(self, num_features, num_classes):
        super(GraphSAGE2, self).__init__()
        self.bn1 = torch.nn.BatchNorm1d(num_features)
        self.linear1 = torch.nn.Linear(num_features, 64)
        self.conv = SAGEConv(64, 64, aggr='mean')
        self.dropout = nn.Dropout(0.2)
        self.bn2 = torch.nn.BatchNorm1d(64)
        self.linear2 = torch.nn.Linear(64, 32)
        self.linear3 = torch.nn.Linear(32, num_classes)


    def forward(self, x, edge_index):
        x = self.bn1(x)
        x = F.leaky_relu(self.linear1(x))
        x = F.leaky_relu(self.conv(x, edge_index))
        x = self.dropout(x)

        x = self.bn2(x)
        x = F.leaky_relu(self.linear2(x))
        x = self.dropout(x)

        x = self.linear3(x)

        return x

In [ ]:
model = GraphSAGE2(num_features=node_features_tensor.size(1), num_classes=2)

decay = 0.05
lr = 0.001

criterion = torch.nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=decay)

train_validate(model=model, data=data, criterion=criterion, optimizer=optimizer, epoch=200, has_edge_weight=False, early_stopping_patience=10)

In [ ]:
test(model, data, criterion, optimizer, has_edge_weight=False, show_cm=True)

#### 5.3 GAT Model
GAT introduces an attention mechanism to the aggregation step, allowing the model to learn the importance (weights) of each neighbor's features dynamically. This means that instead of treating all neighbors equally or relying on pre-determined weights, GAT can focus more on relevant neighbors for each node.

Several important hyperparameters:
1. in_channels (int or tuple) – Size of each input sample, or -1 to derive the size from the first input(s) to the forward method. A tuple corresponds to the sizes of source and target dimensionalities.

2. hidden_channels (int) – Size of each hidden sample.

3. num_layers (int) – Number of message passing layers.

4.  out_channels (int, optional) – If not set to None, will apply a final linear transformation to convert hidden node embeddings to output size out_channels. (default: None)

- `Dimsension: 64`
- `Batch Normalization and feed-forward after input layer`
- `1 GAT layer` with `2 heads`
- `Batch Normalization`
- `Dropout: 0.2`
- `Feed-forward network with 2 linear layers`

In [ ]:
from torch_geometric.nn import GATConv

class GAT1(nn.Module):
    def __init__(self, num_features, num_classes):
        super(GAT1, self).__init__()
        self.bn = torch.nn.BatchNorm1d(num_features)

        self.bn2 = torch.nn.BatchNorm1d(64)
        self.bn3 = torch.nn.BatchNorm1d(64 * 2)

        self.linear1 = torch.nn.Linear(num_features, 64)
        self.layer1 = GATConv(64, 64, heads = 2, dropout = 0.2)

        self.dropout = nn.Dropout(0.2)
        self.linear2 = torch.nn.Linear(64 * 2, 64)
        self.linear3 = torch.nn.Linear(64, num_classes)

    def forward(self, x, edge_index):
        x = self.bn(x)
        x = F.leaky_relu(self.linear1(x))

        x = self.bn2(x)
        x = F.leaky_relu(self.layer1(x, edge_index))
        x = self.dropout(x)

        x = self.bn3(x)
        x = F.leaky_relu(self.linear2(x))
        x = self.dropout(x)

        x = self.linear3(x)
        return x

In [ ]:
model = GAT1(num_features=node_features_tensor.size(1), num_classes=2)

decay = 0.05
lr = 0.001

criterion = torch.nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=decay)

train_validate(model=model, data=data, criterion=criterion, optimizer=optimizer, epoch=200, has_edge_weight=False, early_stopping_patience=10)

In [ ]:
test(model, data, criterion, optimizer, has_edge_weight=False, show_cm=True)